# **Machine Learning for _A Comparative Analysis of Content Across Major Streaming Services_**

#### **Authors**: Karis Park and Elise Huhn

### **Research Questions:**

1. Are there significant differences in genre distribution across streaming platforms?
2. Is there a difference between the platforms in their balance between movies and television shows?
3. Can observations be accurately classified as "Movie" or "TV Show" by their known attributes?

### **Brief Project Overview**

Our first two research questions--"Are there significant differences in genre distribution across streaming platforms?" and "Is there a difference between the platforms in their balance between movies and television shows?"--were able to be be answered through our data analysis. However, our third reseach question--"Can observations be accurately classified as "Movie" or "TV Show" by their known attributes?"--should be explored more thoroughly through machine learning models. Since the variable that we are interested in (movie_or_serie) is a categorical variable with known labels, we will be exploring classification models to predict "movie" or "series" given an observations known attributes. 

* talk about overview of splitting data into training, test, and validation split 

* indicate what machine learning models we will be using and which hyperparameters we will tune using the validation set

### **Machine Learning Package Imports**

In [1]:
# Standard imports -- add others as needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay)

## **Part 1: Preprocessing Data**

In [4]:
#Read in the dataset
streaming_full = pd.read_csv("Data/streaming_combines.csv")
#streaming_full.columns
streaming_full = streaming_full.drop(columns=['title', 'duration_seconds'])
streaming_full.columns

Index(['movie_or_serie', 'release_year', 'channel_streaming', 'action',
       'comedy', 'adventure', 'animation', 'anime', 'drama', 'international',
       'kids', 'horror'],
      dtype='object')

### **Train, Test, and Validation Split**

In [ ]:
#Split data into test and train sets
X = streaming_full.drop(columns=['movie_or_serie'])
y = streaming_full['movie_or_serie']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#Split train data into Validation and Train sets data
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print("------------Split Shapes for Machines Learning--------------------")
print(f"Train:      {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:      {X_test.shape}")

print("\n------------Stratifyed Class Balance Per Split--------------------")
print(f"Class balance (train): {(y_train.value_counts() / len(y_train)).round(3)}")
print(f"Class balance (validation):   {(y_val.value_counts() / len(y_val)).round(3)}")
print(f"Class balance (test):   {(y_test.value_counts() / len(y_test)).round(3)}")

Train:      (14718, 11)
Validation: (3680, 11)
Test:      (4600, 11)
Class balance (train): movie_or_serie
Movie      0.717
TV Show    0.283
Name: count, dtype: float64
Class balance (validation):   movie_or_serie
Movie      0.717
TV Show    0.283
Name: count, dtype: float64
Class balance (test):   movie_or_serie
Movie      0.717
TV Show    0.283
Name: count, dtype: float64


### **Preprocessor**

In [6]:

numeric_features = X_train.select_dtypes(include='number').columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

# Numerical pipeline: impute + standardize
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), #numerical values are replaced with median due to skewdness
    ('scaler',  StandardScaler()), #standardize values for the logistic regression classifier
])

# Categorical pipeline: impute + one-hot encode
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), #categorical values replaced with most frequent due to unknown distributions
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)), #one hot encode categorical vars for ml models
])

# Combine the numerical and categorical pipleines into a preprecessor for whole dataset
preprocessor = ColumnTransformer([
    ('num', numeric_transformer,  numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

### **Choice Justification for Preprocessing and Pipeline**

## **Part 2: Classification Analysis**

### **Choice Justification for Choosing Classification**

### **Classification Model Implmentation**

In [7]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=42)
}

print(f"{'Model':<30} {'AUC mean':>10} {'AUC std':>9} {'Acc mean':>10} {'Acc std':>9}")
print('-' * 73)

for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('clf', clf)])
    auc = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    acc = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    print(f"{name:<30} {auc.mean():>10.3f} {auc.std():>9.3f} {acc.mean():>10.3f} {acc.std():>9.3f}")

Model                            AUC mean   AUC std   Acc mean   Acc std
-------------------------------------------------------------------------
Logistic Regression                 0.754     0.014      0.752     0.008
Random Forest                       0.775     0.013      0.771     0.008
SVM                                 0.709     0.013      0.769     0.008


### **Optimizing Hyperparameters (Via GridSearch)**

In [8]:
#GridSearch for best parameters for the SVM model 
svm_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', SVC(kernel='rbf', probability=True, random_state=42))])

param_grid = {'clf__C': [0.01, 0.1, 1, 10, 100], 
    'clf__gamma': ['scale', 'auto', 0.01, 0.1]}

grid = GridSearchCV(svm_pipe, param_grid, cv=5, scoring='roc_auc_ovr', n_jobs=-1)


grid.fit(X_train, y_train)

print(f"Best parameters: {grid.best_params_}")
print(f"Best CV AUC:     {grid.best_score_:.3f}")

Best parameters: {'clf__C': 100, 'clf__gamma': 0.01}
Best CV AUC:     0.750


In [9]:
#gridSearch for the best hyperparameters for the RF model 
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(random_state=42))
])

param_grid = {
    'clf__n_estimators': [100, 200, 500],
    'clf__max_depth': [None, 5, 10, 20],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(rf_pipe, param_grid, cv=5, scoring='roc_auc_ovr', n_jobs=-1)

grid.fit(X_train, y_train)

print(f"Best parameters: {grid.best_params_}")
print(f"Best CV AUC:     {grid.best_score_:.3f}")

c:\Users\elise\OneDrive\Desktop\SI_618_01\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters: {'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 10, 'clf__n_estimators': 100}
Best CV AUC:     0.794


In [ ]:
logreg_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

param_grid = {
    'clf__penalty': ['l2'],
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__solver': ['lbfgs']
}

grid = GridSearchCV(
    logreg_pipe,
    param_grid,
    cv=5,
    scoring='roc_auc_ovr',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print(f"Best parameters: {grid.best_params_}")
print(f"Best CV AUC:     {grid.best_score_:.3f}")

Best parameters: {'clf__C': 100, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}
Best CV AUC:     0.754


c:\Users\elise\OneDrive\Desktop\SI_618_01\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [12]:
#post-optimized hyperparameter models comparison

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=100, solver = 'lbfgs', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, max_depth=10, max_features="sqrt", min_samples_leaf = 1, min_samples_split = 10,  random_state=42),
    'SVM':                 SVC(kernel='rbf', C=100, gamma=0.01, probability=True, random_state=42) 
}

print(f"{'Model':<30} {'AUC mean':>10} {'AUC std':>9} {'Acc mean':>10} {'Acc std':>9}")
print('-' * 73)

for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('clf', clf)])
    auc = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    acc = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    print(f"{name:<30} {auc.mean():>10.3f} {auc.std():>9.3f} {acc.mean():>10.3f} {acc.std():>9.3f}")


Model                            AUC mean   AUC std   Acc mean   Acc std
-------------------------------------------------------------------------
Logistic Regression                 0.754     0.014      0.752     0.008
Random Forest                       0.794     0.013      0.776     0.005
SVM                                 0.750     0.016      0.770     0.007


### **Choose best model and confusion matrix and auc and accuracy and blah blah blah all that**

### **Results**